# Steenrod operations on symmetric products of surfaces

This notebook records `fastop` computations on symmetric products of closed oriented surfaces. Write Σ_g for the surface of genus g and study

**Symⁿ(Σ_g) = (Σ_g)ⁿ / S_n.**

The main findings are:

- at p = 2, Sym²(Σ_g) detects rank-one Sq²: H² → H⁴ for g = 0, 1, 2, 3;
- at p = 3, Sym³(Σ_g) detects rank-one P¹: H² → H⁶ for g = 0, 1, 2, and in a focused genus-3 run;
- at p = 5, Sym⁵(S²) = CP⁵ and Sym⁵(T²) detect rank-one P¹: H² → H¹⁰;
- at p = 7, Sym⁷(S²) = CP⁷ detects rank-one P¹: H² → H¹⁴ in an opt-in, memory-intensive run;
- compact unordered simplex labels and degree-lazy construction make these large computations practical.


## Setup

Run the notebook from the repository root or from the `notebooks` directory. All standard cells use only the package itself and the Python standard library. The largest positive-genus computations are opt-in.


In [ ]:
from pathlib import Path
from math import comb
import gc
import sys
import time

repo = Path.cwd()
if not (repo / 'src' / 'fastop').exists():
    for candidate in (repo.parent, repo.parent.parent):
        if (candidate / 'src' / 'fastop').exists():
            repo = candidate
            break
sys.path.insert(0, str(repo / 'src'))

from fastop import __version__, spaces

print(f'fastop {__version__}')
print(f'repository: {repo}')


In [ ]:
def timed(function):
    start = time.perf_counter()
    value = function()
    return value, time.perf_counter() - start


def betti_tuple(cohomology, dimension):
    return tuple(cohomology.betti_number(d) for d in range(dimension + 1))


def expected_sym2_betti(genus):
    two_g = 2 * genus
    return (1, two_g, comb(two_g, 2) + 1, two_g, 1)


def expected_sym3_betti(genus):
    two_g = 2 * genus
    return (
        1,
        two_g,
        comb(two_g, 2) + 1,
        comb(two_g, 3) + two_g,
        comb(two_g, 2) + 1,
        two_g,
        1,
    )


## Cohomological ground truth

For a closed oriented surface Σ_g, Macdonald describes `H*(Symⁿ(Σ_g); Z)` as a quotient of the free graded-commutative ring on degree-one generators `x₁,…,x_g,x'₁,…,x'_g` and a degree-two generator `y`. The relations are generated by

```text
x_i₁⋯x_iₐ x'_j₁⋯x'_j_b ∏(y − x_k x'_k) y^q = 0,
a + b + 2c + q = n + 1,
```

with all displayed indices distinct. This presentation is integral and torsion-free. In the unstable range `2 ≤ n ≤ 2g − 2`, one should use the corrected formulation explained in [Gugnin, *On Integral Cohomology Ring of Symmetric Products*](https://arxiv.org/abs/1502.01862), rather than quoting every clause of the original statement literally.

The additive information is compactly encoded by Macdonald's generating function

```text
Σ_n P_t(Symⁿ(Σ_g)) qⁿ = (1 + tq)^(2g) / ((1 − q)(1 − t²q)).
```

For example, when n = 3 the Betti numbers are

```text
(1, 2g, C(2g,2)+1, C(2g,3)+2g, C(2g,2)+1, 2g, 1).
```

For a degree-two class `x`, the instability identities give `Sq²(x) = x²` at p = 2 and `P¹(x) = xᵖ` at odd p. Thus the natural operations in this family are

- p = 2: Sq²: H² → H⁴ on Sym²(Σ_g);
- p = 3: P¹: H² → H⁶ on Sym³(Σ_g);
- p = 5: P¹: H² → H¹⁰ on Sym⁵(Σ_g);
- p = 7: P¹: H² → H¹⁴ on Sym⁷(Σ_g).

The computation below does not insert the divisor class or its power by hand. It constructs cohomology from face maps, evaluates the universal cochain formula on a basis, and projects the result back to cohomology.


## Compact surface models and size preflight

For g > 0, `fastop` uses the standard 4g-gon presentation triangulated by a fan. The surface model has one vertex, 6g − 3 edges, and 4g − 2 triangles.

Before constructing a symmetric power, `symmetric_power_f_vector` counts its nondegenerate simplices using inclusion–exclusion on common degeneracy positions. This is cheap even when the model itself would be too large.


In [ ]:
print('| genus | surface f-vector | Sym² cells | Sym³ cells | Sym⁵ cells |')
print('| ---: | --- | ---: | ---: | ---: |')
for genus in range(4):
    surface = spaces.minimal_simplicial_surface(genus)
    sym2_cells = sum(surface.symmetric_power_f_vector(2))
    sym3_cells = sum(surface.symmetric_power_f_vector(3))
    sym5_cells = sum(surface.symmetric_power_f_vector(5))
    print(
        f'| {genus} | `{surface.f_vector()}` | {sym2_cells:,} | '
        f'{sym3_cells:,} | {sym5_cells:,} |'
    )


The fifth symmetric power changes scale sharply: genus one has 1,797,894 cells, while genus two would have 414,092,094. We therefore compute genus one at p = 5 and treat genus two as a boundary for the current presentation.


## The p = 2 family: Sym²(Σ_g)

The even-prime starting point is both small and informative. The top-square identity gives `Sq²(x) = x²` for `x ∈ H²`, and the computation finds a rank-one map `Sq²: H² → H⁴` for genera 0 through 3. The tested `Sq¹` maps from H¹ and H² vanish. For g = 0 this is the familiar `Sym²(S²) = CP²` calculation; positive genus checks the same operation directly on substantially different quotient models.


In [ ]:
p2_results = []
for genus in range(4):
    model, build_seconds = timed(
        lambda genus=genus: spaces.symmetric_product_of_surface(genus, 2)
    )
    cohomology = model.cohomology(p=2)
    measured_betti, cohomology_seconds = timed(
        lambda: betti_tuple(cohomology, model.dimension)
    )
    sq2_rank, operation_seconds = timed(
        lambda: cohomology.operation_rank(2, 2)
    )
    assert measured_betti == expected_sym2_betti(genus)
    assert cohomology.operation_rank(1, 1) == 0
    assert cohomology.operation_rank(2, 1) == 0
    assert sq2_rank == 1
    p2_results.append(
        (genus, sum(model.f_vector()), measured_betti, build_seconds,
         cohomology_seconds, operation_seconds)
    )

print('| genus | cells | Betti numbers | build | cohomology | Sq² rank/time |')
print('| ---: | ---: | --- | ---: | ---: | --- |')
for genus, cells, betti, build_s, cohom_s, operation_s in p2_results:
    print(
        f'| {genus} | {cells:,} | `{betti}` | {build_s:.3f}s | '
        f'{cohom_s:.3f}s | 1 / {operation_s:.3f}s |'
    )


## The p = 3 family: Sym³(Σ_g)

The standard run computes full mod-3 cohomology for genera 0, 1, and 2 and compares it with Macdonald's formula.


In [ ]:
p3_results = []
for genus in (0, 1, 2):
    model, build_seconds = timed(
        lambda genus=genus: spaces.symmetric_product_of_surface(genus, 3)
    )
    cohomology = model.cohomology(p=3)
    measured_betti, cohomology_seconds = timed(
        lambda: betti_tuple(cohomology, model.dimension)
    )
    rank, operation_seconds = timed(lambda: cohomology.operation_rank(2, 1))
    assert measured_betti == expected_sym3_betti(genus)
    assert rank == 1
    p3_results.append(
        (genus, sum(model.f_vector()), measured_betti, build_seconds,
         cohomology_seconds, operation_seconds)
    )
    del cohomology, model
    gc.collect()

print('| genus | cells | Betti numbers | build | cohomology | P¹ rank/time |')
print('| ---: | ---: | --- | ---: | ---: | --- |')
for genus, cells, betti, build_s, cohom_s, operation_s in p3_results:
    print(
        f'| {genus} | {cells:,} | `{betti}` | {build_s:.3f}s | '
        f'{cohom_s:.3f}s | 1 / {operation_s:.3f}s |'
    )


### The first unstable case

Genus three is the first case in the unstable range 2 ≤ n ≤ 2g − 2 for n = 3. Its model has 189,626 cells and the focused computation finds rank-one P¹. Full middle-degree cohomology is intentionally optional because that linear algebra is considerably more expensive.


In [ ]:
RUN_GENUS_THREE = False

if RUN_GENUS_THREE:
    sym3_genus3, build_seconds = timed(
        lambda: spaces.symmetric_product_of_surface(3, 3)
    )
    H = sym3_genus3.cohomology(p=3)
    rank, operation_seconds = timed(lambda: H.operation_rank(2, 1))
    assert rank == 1
    print(sym3_genus3.f_vector())
    print(f'build: {build_seconds:.3f}s; P1 rank: {rank}; operation: {operation_seconds:.3f}s')
else:
    print('Set RUN_GENUS_THREE = True for the extended genus-3 computation.')


## The p = 5 family: Sym⁵(Σ_g)

The sphere gives the projective-space ground truth Sym⁵(S²) = CP⁵. This standard cell is safe to run and verifies the full mod-5 Betti vector before evaluating P¹.


In [ ]:
CP5, build_seconds = timed(lambda: spaces.symmetric_product_of_surface(0, 5))
H_CP5 = CP5.cohomology(p=5)
cp5_betti, cohomology_seconds = timed(lambda: betti_tuple(H_CP5, 10))
cp5_rank, operation_seconds = timed(lambda: H_CP5.operation_rank(2, 1))

assert cp5_betti == (1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1)
assert cp5_rank == 1
print(f'cells: {sum(CP5.f_vector()):,} {CP5.f_vector()}')
print(f'Betti numbers: {cp5_betti}')
print(
    f'build: {build_seconds:.3f}s; cohomology: {cohomology_seconds:.3f}s; '
    f'P1 rank: {cp5_rank}; operation: {operation_seconds:.3f}s'
)


### Positive genus at p = 5

Sym⁵(T²) is the first positive-genus example. Topologically it is a CP⁴-bundle over the torus, but the computation below works directly with the normalized symmetric simplicial set.

Reference result: 1,797,894 cells, b₂ = 2, b₁₀ = 1, and rank-one P¹. The cold focused pipeline takes roughly 18 seconds and peaks near 780 MiB on the development machine, so it is opt-in.


In [ ]:
RUN_LARGE_P5 = False

if RUN_LARGE_P5:
    sym5_torus, build_seconds = timed(
        lambda: spaces.symmetric_product_of_surface(1, 5)
    )
    H = sym5_torus.cohomology(p=5)
    b2, b2_seconds = timed(lambda: H.betti_number(2))
    b10, b10_seconds = timed(lambda: H.betti_number(10))
    rank, operation_seconds = timed(lambda: H.operation_rank(2, 1))
    assert (b2, b10, rank) == (2, 1, 1)
    print(f'cells: {sum(sym5_torus.f_vector()):,}')
    print(
        f'build: {build_seconds:.3f}s; b2: {b2} in {b2_seconds:.3f}s; '
        f'b10: {b10} in {b10_seconds:.3f}s; P1 rank: {rank} '
        f'in {operation_seconds:.3f}s'
    )
else:
    print('Set RUN_LARGE_P5 = True for the 1.8-million-cell torus run.')


## The p = 7 endpoint: Sym⁷(S²) = CP⁷

At the next odd prime, the projective-space ground truth is still reachable: `P¹: H²(CP⁷; F₇) → H¹⁴(CP⁷; F₇)` has rank one. The complete normalized model has 13,478,264 cells, but degree-lazy symmetric powers avoid constructing all of them.

> **Resource warning.** This cell is disabled by default. On the development arm64 macOS machine, computing H¹⁴ took 72.4 seconds, evaluating P¹ took 24.9 seconds, and the process peaked at 1.76 GiB. Close other memory-intensive applications before enabling it. This is an opt-in benchmark, not a routine notebook cell.


In [ ]:
RUN_LARGE_P7 = False

if RUN_LARGE_P7:
    CP7, build_seconds = timed(
        lambda: spaces.symmetric_product_of_surface(0, 7)
    )
    H = CP7.cohomology(p=7)
    b2, b2_seconds = timed(lambda: H.betti_number(2))
    b14, b14_seconds = timed(lambda: H.betti_number(14))
    rank, operation_seconds = timed(lambda: H.operation_rank(2, 1))
    assert (b2, b14, rank) == (1, 1, 1)
    print(f'predicted cells: {sum(CP7.f_vector()):,}')
    print(
        f'lazy build: {build_seconds:.3f}s; b2: {b2} in {b2_seconds:.3f}s; '
        f'b14: {b14} in {b14_seconds:.3f}s; P1 rank: {rank} '
        f'in {operation_seconds:.3f}s'
    )
else:
    print('Set RUN_LARGE_P7 = True for the ~1.76-GiB CP7 computation.')


## What made the large computations possible

Three implementation choices determine the practical range:

1. **Degeneracy-capacity pruning.** A partial symmetric tuple is abandoned when its remaining factors cannot remove all common degeneracy positions.
2. **Compact unordered labels.** Symmetric cells are stored as sorted tuples of small reference indices; faces are computed only when requested.
3. **Degree-lazy models and cohomology.** Symmetric-product cells and boundary data are materialized only in requested degrees. A focused P¹ computation works around degrees 2 and 10 at p = 5, or degrees 2 and 14 at p = 7, without constructing the enormous middle-dimensional layers.

The size preflight then tells us when even this representation is no longer appropriate. For Sym⁵(Σ₂), 414 million cells rule out a direct construction before any memory is allocated.


## Conclusions and next search

The symmetric products of oriented surfaces provide a coherent progression through the primes:

- Sym²(S²) = CP² validates p = 2; increasing genus preserves a rank-one Sq²;
- Sym³(S²) = CP³ validates p = 3; increasing genus tests complexity while preserving a rank-one P¹;
- Sym⁵(S²) = CP⁵ validates p = 5; Sym⁵(T²) shows that the implementation reaches a genuinely large positive-genus quotient;
- Sym⁷(S²) = CP⁷ validates p = 7 with rank-one P¹, but its 1.76-GiB peak marks the present laptop-scale boundary.

The next efficient p = 5 search should not immediately increase surface genus. It should screen other compact ten-dimensional geometric or combinatorial models by cohomology algebra and cell count, then evaluate P¹ only on candidates with a plausible nonzero H² → H¹⁰ target.


## Bonus direction: non-orientable surfaces

There is a clean classification for the cohomology ring of the surface itself. If `N_h = #^h RP²`, then over F₂ one can choose degree-one classes `a₁,…,a_h` and the degree-two fundamental class `u` so that

```text
a_i a_j = 0  (i ≠ j),    a_i² = u,
```

with all products above degree two zero. Thus non-orientability is already visible through nonzero squares of degree-one classes, making `Sq¹(a_i) = a_i²` an especially attractive computational target.

For `Symⁿ(N_h)`, the situation is less uniform than Macdonald's torsion-free presentation for oriented surfaces. Integral 2-torsion is essential, and at p = 2 the quotient order is not invertible, so invariant-theory shortcuts do not apply. There is nevertheless a general multiplicative cellular model for symmetric products of finite two-dimensional complexes; see [Kallel–Salvatore, *Symmetric Products of Two Dimensional Complexes*](https://arxiv.org/abs/math/0510176). It gives an effective route to the additive groups and products, but not a single Macdonald-style presentation that we should quote without qualifications for every `h` and `n`.

For fastop, the sensible bonus experiment is therefore small and computational: add compact models of `N_h`, begin with `RP²` and the Klein bottle, construct `Sym²`, and compare `Sq¹` and `Sq²` against the cellular calculation before attempting a general family. This should remain separate from the oriented-surface showcase.
